**Nota**: Este tutorial es una adaptación de https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/quick_start_with_hugging_face.ipynb

## Configuración

Para completar este tutorial, necesitarás un entorno de ejecución con [recursos suficientes](https://ai.google.dev/gemma/docs/core#sizes) para ejecutar el modelo **MedGemma**.

Puedes probar **MedGemma 4B** de forma gratuita en Google Colab utilizando una **GPU T4**:

1. En la esquina superior derecha de la ventana de Colab, selecciona **▾ (Opciones de conexión adicionales)**.  
2. Selecciona **Cambiar tipo de entorno de ejecución**.  
3. En **Acelerador de hardware**, selecciona **GPU T4**.

**Nota**: Para ejecutar la demostración con **MedGemma 27B** en Google Colab, necesitarás un entorno de ejecución con una **GPU A100** y usar **cuantización de 4 bits** para reducir el uso de memoria. El rendimiento de las versiones cuantizadas no ha sido evaluado.

### Obtener acceso a MedGemma

Antes de comenzar, asegúrate de tener acceso a los modelos **MedGemma** en **Hugging Face**:

1. Si aún no tienes una cuenta en Hugging Face, puedes crear una de forma gratuita haciendo clic [aquí](https://huggingface.co/join).  
2. Dirígete a la [página del modelo MedGemma](https://huggingface.co/google/medgemma-4b-it) y acepta las condiciones de uso.

### Autenticarse con Hugging Face

Genera un token de acceso de tipo `read` en Hugging Face yendo a la sección de [configuración](https://huggingface.co/settings/tokens).

Si estás usando **Google Colab**, agrega tu token de acceso al **administrador de secretos de Colab** para almacenarlo de forma segura. Si no es así, simplemente ejecuta la celda que aparece más abajo para autenticarte con Hugging Face.

1. Abre tu cuaderno de Google Colab y haz clic en la pestaña **🔑 Secrets** ubicada en el panel izquierdo.  
   <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="La pestaña Secrets se encuentra en el panel izquierdo." width=50%>  
2. Crea un nuevo secreto con el nombre `HF_TOKEN`.  
3. Copia y pega tu clave de token en el campo **Value** de `HF_TOKEN`.  
4. Activa el botón del lado izquierdo para permitir que el cuaderno acceda al secreto.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    from huggingface_hub import get_token, notebook_login
    if get_token() is None:
        notebook_login()

### Instalar dependencias

In [5]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

## Cargar modelo desde Hugging Face Hub

In [ ]:
from transformers import BitsAndBytesConfig
import torch

model_id = "google/medgemma-4b-it"

model_kwargs = dict(
    dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)

**Cargar el modelo con la API `pipeline`**

In [ ]:
from transformers import pipeline

pipe = pipeline("image-text-to-text", model=model_id, model_kwargs=model_kwargs)
pipe.model.generation_config.do_sample = False

## Correr inferencia sobre imágenes y texto

Esta sección muestra cómo ejecutar inferencias en tareas basadas en imágenes.

**Especificar las entradas de imagen y texto**

In [ ]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display, Markdown

prompt = "Describe this X-ray"  # @param {type: "string"}

# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"  # @param {type: "string"}
! wget -nc -q {image_url}
image_filename = os.path.basename(image_url)
image = Image.open(image_filename)

**Conversión de formato**

In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are an expert radiologist."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": image}
        ]
    }
]
max_new_tokens = 300

**Correr el modelo con la API `pipeline`**

In [ ]:
output = pipe(text=messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

# Probando diferentes prompts

Primero descargamos 5 imágenes:

In [ ]:
!gdown 1nQLsqbYCR9N6qAm2l2WdhLEGBEWg4-eE
!gdown 1xwfatIX3z9jHDelRYB7AOL58Chr37dzY
!gdown 16burJBGgndpiDG0AbX3Ky27EzAtH6PAT
!gdown 1pkxb83uOv0RTY0VLSpDcwcfbq6pjp24U
!gdown 1bBx6kA699O-qZYXtXkHwXETTcV8BgJZy
!gdown 1EX-EOWCJlZrEam3BwAbkWlOcS64W2XkC


Downloading...
From: https://drive.google.com/uc?id=16burJBGgndpiDG0AbX3Ky27EzAtH6PAT
To: /content/padchest_abnormal_2.jpg
100% 178k/178k [00:00<00:00, 109MB/s]


In [ ]:
def generate_text_from_image(
    pipe,
    prompt: str,
    image_path: str,
    system_instruction: str = "You are an expert radiologist.",
    max_new_tokens: int = 600,
):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_instruction}]},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image", "image": image_path},
            ],
        },
    ]

    output = pipe(text=messages, max_new_tokens=max_new_tokens)
    response = output[0]["generated_text"][-1]["content"]

    display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
    display(IPImage(filename=image_path, height=300))
    display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

    return response.strip()

# Ejemplo 1: padchest_normal.jpg

In [ ]:
prompt = "Describe this X-ray"
image_path = "/content/padchest_normal.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

In [ ]:
prompt = "Generate a radiology report for this X-ray. Write findings first and then impression."
image_path = "/content/padchest_normal.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

# Ejemplo 2: padchest_abnormal_1.jpg



In [ ]:
prompt = "Describe this X-ray"
image_path = "/content/padchest_abnormal_1.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

In [ ]:
prompt = "Generate a radiology report for this X-ray. Write findings first and then impression."
image_path = "/content/padchest_abnormal_1.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

In [ ]:
prompt = "Generate a radiology report for this X-ray. Write findings first and then impression. Write briefly and concisely."
image_path = "/content/padchest_abnormal_1.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

# Ejemplo 3: padchest_abnormal_2.jpg

In [ ]:
prompt = "Describe this X-ray"
image_path = "/content/padchest_abnormal_2.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

In [ ]:
prompt = "Enumerate positive findings, enumerate negative findings, then write a final impression."
image_path = "/content/padchest_abnormal_2.jpg"
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)

# Ejercicio libre: escribe tus propios prompts

In [ ]:
prompt = "Describe this X-ray"  # Cambia el prompt aquí
image_path = "/content/padchest_abnormal_2.jpg"  # Opciones: padchest_normal.jpg, padchest_abnormal_1.jpg, padchest_abnormal_2.jpg
system_instruction = "You are an expert radiologist"

result = generate_text_from_image(
    pipe=pipe,
    prompt=prompt,
    image_path=image_path,
    system_instruction=system_instruction,
)